In [16]:
import pandas as pd
import yaml

In [17]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [18]:
config

{'data': {'raw': {'file1': '../data/raw/df_final_demo.txt',
   'file2': '../data/raw/df_final_experiment_clients.txt',
   'file3': '../data/raw/df_final_web_data_pt_1.txt',
   'file4': '../data/raw/df_final_web_data_pt_2.txt',
   'file5': '../data/raw/final_web_data_merged.csv'},
  'clean': {'file1': '../data/clean/df_demo_cleaned.csv',
   'file2': '../data/clean/df_experiment_clients_cleaned.csv',
   'file3': '../data/clean/df_test_cids.csv',
   'file4': '../data/clean/df_control_cids.csv',
   'file5': '../data/clean/final_web_data_merged_confirmed.csv',
   'file6': '../data/clean/final_web_data_merged_cleaned.csv'}}}

In [41]:
df_d = pd.read_csv(config['data']['raw']['file1'], quotechar='"')
df_c = pd.read_csv(config['data']['raw']['file2'], quotechar='"')

In [6]:
def clean_demo(df):
    df2 = df.copy()
    df2.dropna(thresh=2)
    return df2

def find_age_tenure_mismatch(df):
    df2 = df.copy()
    df2 = df2[df2.clnt_age> df2.clnt_tenure_yr]
    return df2

def clean_clients_dropna(df):
    df2 = df.copy()
    df2.rename(str.lower, axis="columns")
    df2 = df2.dropna()
    return df2

def find_children(df):
    df2 = df.copy()
    df2 = df2[df2.clnt_age<18]
    return df2

In [42]:
df_d = clean_demo(df_d)
df_c = clean_clients_dropna(df_c)
df_c.nunique()

client_id    50500
Variation        2
dtype: int64

In [40]:
df_c.nunique()

client_id    50500
Variation        2
dtype: int64

In [14]:
df_control = df_c[df_c.Variation=='Control']
df_test = df_c[df_c.Variation=='Test']

,client_id,Variation
2,4033851,Control
4,9294070,Control
7,6651403,Control
9,2105948,Control
12,9814849,Control
...,...,...
50491,4364429,Control
50493,8730282,Control
50494,5305116,Control
50495,393005,Control


In [15]:
df_d.to_csv("../data/clean/df_demo_cleaned.csv", index=False, encoding= "utf-8", sep = ",")
df_c.to_csv("../data/clean/df_experiment_clients_cleaned.csv", index=False, encoding= "utf-8", sep = ",")
df_control.to_csv("../data/clean/df_control_cids.csv", index=False, encoding= "utf-8", sep = ",")
df_test.to_csv("../data/clean/df_test_cids.csv", index=False, encoding= "utf-8", sep = ",")

In [52]:
#checked for null values
#df_d.isna().sum()

In [53]:
#inspected rows with nulls
#df_d[df_d.isna().any(axis=1)]

In [54]:
#trying to understand why so many children appear
#df_d[df_d.clnt_age<18]

In [55]:
#noticed many children had tenures longer than their lives
#df_d[df_d.clnt_age<df_d.clnt_tenure_yr]

In [56]:
#confirmed no errors in tenure month vs year
# df_d[df_d.clnt_tenure_mnth> (df_d.clnt_tenure_yr +1)*12]
 # df_d[df_d.clnt_tenure_mnth< (df_d.clnt_tenure_yr )*12]

In [57]:
#inspect values for errors
# df_d.describe()

In [58]:
#confirm no duplicates
#df_d.client_id.nunique(), df_d.client_id.count()

In [59]:
#look for empty values
#df_c.isna().sum()

In [60]:
# check if the proportion for test groups is approximately equal
#df_c.Variation.value_counts(normalize=True)

In [61]:
# looking at missing values
#df_c[df_c.isna().any(axis=1)]

In [62]:
#check if there are duplicates and what they are
#df_c.client_id.nunique(), df_c.client_id.count()

In [71]:
df_w = pd.read_csv(config['data']['clean']['file6'], quotechar='"')

In [72]:
df_w["date_time"] = pd.to_datetime(df_w["date_time"])

In [74]:
remove_id=set()
for client_id, group in df_w.groupby('client_id'):
    group.sort_values(by='date_time', ascending=False)
    if group.visit_id.nunique()>1:
        visit_list = group['visit_id'].unique().tolist()
        #print(group.visit_id.iloc[0])
        visit_list.remove(group.visit_id.iloc[0])
        remove_id.update(visit_list)
        

In [75]:
df_w = df_w[~df_w.visit_id.isin(remove_id)]

In [76]:
#df_w[df_w.visit_id=='151735117_75089245092_170146']

,client_id,visitor_id,visit_id,process_step,date_time
38057,1237509,278282875_50288201923,151735117_75089245092_170146,step_3,2017-06-14 08:36:27
38058,1237509,278282875_50288201923,151735117_75089245092_170146,step_2,2017-06-14 08:35:55
38059,1237509,278282875_50288201923,151735117_75089245092_170146,step_1,2017-06-14 08:35:41
38060,1237509,278282875_50288201923,151735117_75089245092_170146,start,2017-06-14 08:35:34


In [77]:
#df_w[df_w.client_id==1237509]

,client_id,visitor_id,visit_id,process_step,date_time
38057,1237509,278282875_50288201923,151735117_75089245092_170146,step_3,2017-06-14 08:36:27
38058,1237509,278282875_50288201923,151735117_75089245092_170146,step_2,2017-06-14 08:35:55
38059,1237509,278282875_50288201923,151735117_75089245092_170146,step_1,2017-06-14 08:35:41
38060,1237509,278282875_50288201923,151735117_75089245092_170146,start,2017-06-14 08:35:34


In [78]:
df_w.to_csv("../data/clean/df_merged_unique_vids.csv", index=False, encoding= "utf-8", sep = ",")